In [ ]:
import pandas as pd

In [ ]:
LEVEL = 'ageXclass'

In [ ]:
# This one has PAS name - group info

df_sample_merged_info = pd.read_pickle('./PAS_merged_from_many_samples_info.pkl')
df_sample_merged_info

In [ ]:
# This one has all PAS info

df_gene_dup_info = pd.read_pickle(f'./PAS_duplicated_from_many_genes_removal_info-{LEVEL}.pkl')
df_gene_dup_info

In [ ]:
# Add cell type to names

df_gene_dup_info['nameWithCellType'] = df_gene_dup_info['name'] + '|' + df_gene_dup_info['cellType']

In [ ]:
df_gene_dup_info[df_gene_dup_info['nameWithCellType']=='PYURF|1|PYURF|coding|ENST00000273968|3UTR|Middle_adult-EN']

In [ ]:
df_gene_dup_info[(df_gene_dup_info.chrom=='4') &
(df_gene_dup_info.start==88521055) &
(df_gene_dup_info.end==88521411)]

In [ ]:
# If these colums are the same, duplciates exists in df_gene_dup_info but not in df_sample_merged_info

peak_identifiers = 'chrom start end strand blockSizes blockStarts'.split()

In [ ]:
# So, this should be almost 450 and 0

print((len(df_gene_dup_info.drop_duplicates(peak_identifiers + ['cellType'])) - len(df_gene_dup_info)) / df_gene_dup_info['cellType'].nunique())
print(len(df_gene_dup_info.drop_duplicates(peak_identifiers + ['cellType'])) - len(df_sample_merged_info))

# Map every PAS group to every genes

In [ ]:
# df_sample_merged_info has name (of selected PAS) - group info
# df_gene_dup_info has name - location info and can merge names

In [ ]:
# Takes < 1 min

gb_pas = df_gene_dup_info.groupby(peak_identifiers + ['cellType'])
se_mapped_genes = gb_pas['geneName'].apply(lambda x: list(x))
se_repr_pas_name = gb_pas['nameWithCellType'].first()

df_pas_duplicated_over_genes = pd.DataFrame()
df_pas_duplicated_over_genes['reprName'] = se_repr_pas_name
df_pas_duplicated_over_genes['geneNameList'] = se_mapped_genes

df_pas_duplicated_over_genes

In [ ]:
(df_pas_duplicated_over_genes['geneNameList'].apply(len) > 1).sum()

In [ ]:
# Priority between PYURF and PIGY couldn't be set because they are tie.

set(df_sample_merged_info['name']) - set(df_pas_duplicated_over_genes['reprName'])

In [ ]:
se_pas_group_to_all_overlapped_genes = pd.merge(df_sample_merged_info, df_pas_duplicated_over_genes, 
                                                left_on='name', right_on='reprName').groupby('group')['geneNameList'].sum().apply(lambda x: sorted(set(x)))

In [ ]:
df_pas_group_to_all_overlapped_genes = se_pas_group_to_all_overlapped_genes.apply(lambda x: ';'.join(x)).reset_index()
df_pas_group_to_all_overlapped_genes.to_csv('PAS_group_to_all_overlapped_genes.tsv.gz', sep='\t', compression='gzip')

df_pas_group_to_all_overlapped_genes = se_pas_group_to_all_overlapped_genes.reset_index()
df_pas_group_to_all_overlapped_genes.to_pickle('PAS_group_to_all_overlapped_genes.pkl')

In [ ]:
from collections import Counter
Counter(df_pas_group_to_all_overlapped_genes['geneNameList'].apply(len))

In [ ]:
df_pas_group_to_all_overlapped_genes[df_pas_group_to_all_overlapped_genes['geneNameList'].apply(len)>=2]